In [1]:
! pip install googletrans==4.0.0-rc1 bs4

In [3]:
import pandas as pd
import os

df = pd.read_parquet("hf://datasets/gentaiscool/codemixqa/data/test-00000-of-00001.parquet")

In [4]:
df

,id,original_index,problem,answer,topic,answer_type,multi_step,requires_reasoning,urls,method,language
0,2174-grammarforce_source-Bengali,2174,Which baseball field-এ Central Illinois Colleg...,Lloyd Hopkins Field,Sports,Place,False,False,https://www.altonbaseball.com/custom_pages/982...,grammarforce_source,Bengali
1,2429-grammarforce_target-Urdu,2429,Tivoli Gardens میں Demon کے لیے جگہ بنانے کو ک...,The Snake,Other,Other,False,False,https://sophiessuitcase.com/a-guide-to-tivoli-...,grammarforce_target,Urdu
2,3532-grammarforce_source-Hindi,3532,1898 में New York के state entomologist की 14t...,Byturus tomentosus,Science and technology,Other,True,False,https://nysl.ptfs.com/aw-server/rest/product/p...,grammarforce_source,Hindi
3,1452-selective-Italian,1452,After chi was Bullard Mountain named in the Bo...,Benjamin Bullard,Geography,Person,False,False,https://en.wikipedia.org/wiki/Bullard_Mountain...,selective,Italian
4,4184-force-Japanese,4184,What four 健康組織 wrote to President ジョン・F・ケネディ c...,"American Cancer Society, the American Public H...",History,Other,False,False,https://www.ajmc.com/view/surgeon-generals-smo...,random,Japanese
...,...,...,...,...,...,...,...,...,...,...,...
63995,4283-selective-Urdu transliterate,4283,kis month aur year mein Naughty Dog ke technol...,November 2023,Video games,Date,False,False,"https://en.wikipedia.org/wiki/Naughty_Dog,http...",selective,Urdu transliterate
63996,4287-selective-Urdu transliterate,4287,Which English singer aik film aur television c...,George Melly,Music,Person,False,False,https://tv.apple.com/us/person/george-melly/um...,selective,Urdu transliterate
63997,4291-selective-Urdu transliterate,4291,"MI5 ki pehli female DG kaun thi, aur pehla DG ...",Stella Rimington,Politics,Person,False,False,https://www.horus-security.co.uk/articles/nota...,selective,Urdu transliterate
63998,4306-selective-Urdu transliterate,4306,William Holwell Carr ke walid kya karte the zi...,Apothecary,Art,Other,False,False,https://en.wikipedia.org/wiki/William_Holwell_...,selective,Urdu transliterate


Created a folder with a dump of all the unique urls for each language

In [4]:
import pandas as pd

# Build a unique URL list and map each URL to a numeric id
all_urls = (
    df["urls"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)
all_urls = all_urls[all_urls != ""]
unique_urls = pd.Series(all_urls.unique()).sort_values().reset_index(drop=True)

url_id_map = {url: idx for idx, url in enumerate(unique_urls, start=1)}

# Add columns:
# - url_list: list of URLs per row
# - url_ids: list of numeric ids corresponding to each URL
df["url_list"] = (
    df["urls"]
    .fillna("")
    .apply(lambda x: [u.strip() for u in x.split(",") if u.strip()])
)

def map_urls_to_ids(urls):
    return [url_id_map[u] for u in urls if u in url_id_map]

df["url_ids"] = df["url_list"].apply(map_urls_to_ids)

print(f"Total unique URLs: {len(unique_urls):,}")
print(df[["urls", "url_ids"]].head())

Total unique URLs: 2,939
                                                urls                url_ids
0  https://www.altonbaseball.com/custom_pages/982...            [1981, 908]
1  https://sophiessuitcase.com/a-guide-to-tivoli-...      [1768, 170, 1225]
2  https://nysl.ptfs.com/aw-server/rest/product/p...           [1608, 2339]
3  https://en.wikipedia.org/wiki/Bullard_Mountain...         [516, 334, 82]
4  https://www.ajmc.com/view/surgeon-generals-smo...  [1967, 232, 2565, 74]


In [5]:
# from collections import defaultdict

# dict_lang_ids = defaultdict(set)

# for _, row in df.iterrows():
#     lang = row["language"]
#     ids = row["url_ids"]
#     for id in ids:
#         dict_lang_ids[lang].add(id)

# # convert sets to sorted lists
# dict_lang_ids = {
#     lang: sorted(list(ids))
#     for lang, ids in dict_lang_ids.items()
# }

# print(dict_lang_ids)

# for key, val in dict_lang_ids.items():
#     print(key)
#     print(len(val))

In [6]:
df

,id,original_index,problem,answer,topic,answer_type,multi_step,requires_reasoning,urls,method,language,url_list,url_ids
0,2174-grammarforce_source-Bengali,2174,Which baseball field-এ Central Illinois Colleg...,Lloyd Hopkins Field,Sports,Place,False,False,https://www.altonbaseball.com/custom_pages/982...,grammarforce_source,Bengali,[https://www.altonbaseball.com/custom_pages/98...,"[1981, 908]"
1,2429-grammarforce_target-Urdu,2429,Tivoli Gardens میں Demon کے لیے جگہ بنانے کو ک...,The Snake,Other,Other,False,False,https://sophiessuitcase.com/a-guide-to-tivoli-...,grammarforce_target,Urdu,[https://sophiessuitcase.com/a-guide-to-tivoli...,"[1768, 170, 1225]"
2,3532-grammarforce_source-Hindi,3532,1898 में New York के state entomologist की 14t...,Byturus tomentosus,Science and technology,Other,True,False,https://nysl.ptfs.com/aw-server/rest/product/p...,grammarforce_source,Hindi,[https://nysl.ptfs.com/aw-server/rest/product/...,"[1608, 2339]"
3,1452-selective-Italian,1452,After chi was Bullard Mountain named in the Bo...,Benjamin Bullard,Geography,Person,False,False,https://en.wikipedia.org/wiki/Bullard_Mountain...,selective,Italian,[https://en.wikipedia.org/wiki/Bullard_Mountai...,"[516, 334, 82]"
4,4184-force-Japanese,4184,What four 健康組織 wrote to President ジョン・F・ケネディ c...,"American Cancer Society, the American Public H...",History,Other,False,False,https://www.ajmc.com/view/surgeon-generals-smo...,random,Japanese,[https://www.ajmc.com/view/surgeon-generals-sm...,"[1967, 232, 2565, 74]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
63995,4283-selective-Urdu transliterate,4283,kis month aur year mein Naughty Dog ke technol...,November 2023,Video games,Date,False,False,"https://en.wikipedia.org/wiki/Naughty_Dog,http...",selective,Urdu transliterate,"[https://en.wikipedia.org/wiki/Naughty_Dog, ht...","[1002, 60, 2930, 2296]"
63996,4287-selective-Urdu transliterate,4287,Which English singer aik film aur television c...,George Melly,Music,Person,False,False,https://tv.apple.com/us/person/george-melly/um...,selective,Urdu transliterate,[https://tv.apple.com/us/person/george-melly/u...,"[1847, 689, 1569]"
63997,4291-selective-Urdu transliterate,4291,"MI5 ki pehli female DG kaun thi, aur pehla DG ...",Stella Rimington,Politics,Person,False,False,https://www.horus-security.co.uk/articles/nota...,selective,Urdu transliterate,[https://www.horus-security.co.uk/articles/not...,"[2387, 1185, 1801]"
63998,4306-selective-Urdu transliterate,4306,William Holwell Carr ke walid kya karte the zi...,Apothecary,Art,Other,False,False,https://en.wikipedia.org/wiki/William_Holwell_...,selective,Urdu transliterate,[https://en.wikipedia.org/wiki/William_Holwell...,"[1276, 2566]"


In [45]:
# now store the unique urls in a file 
unique_urls.to_csv("unique_urls.csv", index=False)

In [46]:
# storing the mapping as well
url_id_df = pd.DataFrame({
    "url": list(url_id_map.keys()),
    "url_id": list(url_id_map.values())
})
url_id_df.to_csv("url_id_map.csv", index=False)

In [47]:
import requests
from bs4 import BeautifulSoup

# for each unique url, parse it and store it in a {id}.txt file

output_dir_lang = "language_texts"
os.makedirs(output_dir_lang, exist_ok=True)
def fetch_url_text(url, timeout=10):
    try:
        resp = requests.get(url, timeout=timeout, headers={
            "User-Agent": "Mozilla/5.0"
        })
        resp.raise_for_status()

        soup = BeautifulSoup(resp.text, "html.parser")

        # Remove scripts/styles
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(separator=" ")
        text = " ".join(text.split())  # normalize whitespace
        return text

    except Exception as e:
        print(f"Failed to fetch {url}: {e}")
        return None

# tqdm for progress bar
from tqdm import tqdm
for url, url_id in tqdm(url_id_map.items()):
    text = fetch_url_text(url)
    if text:
        with open(os.path.join(output_dir_lang, f"{url_id}.txt"), "w", encoding="utf-8") as f:
            f.write(text)

  0%|          | 0/2939 [00:00<?, ?it/s]

Failed to fetch 'https://en.wikipedia.org/wiki/Ben_M._Bogard': No connection adapters were found for "'https://en.wikipedia.org/wiki/Ben_M._Bogard'"
Failed to fetch 'https://en.wikipedia.org/wiki/Lygia_Pape': No connection adapters were found for "'https://en.wikipedia.org/wiki/Lygia_Pape'"
Failed to fetch 'https://en.wikipedia.org/wiki/Omsund_Bridge': No connection adapters were found for "'https://en.wikipedia.org/wiki/Omsund_Bridge'"
Failed to fetch 'https://en.wikipedia.org/wiki/S%C3%BCleymaniye_Mosque': No connection adapters were found for "'https://en.wikipedia.org/wiki/S%C3%BCleymaniye_Mosque'"
Failed to fetch Hillhttps://en.wikipedia.org/wiki/Margaret_Gardiner_(art_collector): No connection adapters were found for 'Hillhttps://en.wikipedia.org/wiki/Margaret_Gardiner_(art_collector)'
Failed to fetch chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://www.ams.org/notices/199808/comm-fulkerson.pdf: No connection adapters were found for 'chrome-extension://efaidnbmnnnibpca

  0%|          | 7/2939 [00:00<01:49, 26.90it/s]

Failed to fetch http://astro.vaporia.com/start/granat.html: 406 Client Error: Not Acceptable for url: http://astro.vaporia.com/start/granat.html


  0%|          | 12/2939 [00:01<09:10,  5.31it/s]

...

100%|█████████▉| 2934/2939 [44:21<00:03,  1.67it/s]

Failed to fetch https://ysk.gov.tr/doc/dosyalar/Ingilizce/ElectionResults/2018CB-416D_en.pdf: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


100%|█████████▉| 2937/2939 [44:25<00:02,  1.08s/it]

Failed to fetch https://zenodo.org/records/6182816: 403 Client Error: Forbidden for url: https://zenodo.org/records/6182816


100%|██████████| 2939/2939 [44:26<00:00,  1.10it/s]


In [8]:
# printing the number of successfully fetched files
output_dir_lang = "language_texts"
fetched_files = len(os.listdir(output_dir_lang))
print(f"Fetched and stored {fetched_files} documents out of {len(unique_urls)}")

Fetched and stored 2419 documents out of 2939


In [9]:
set_langs = set(df['language'].dropna().unique())
print(set_langs)

{'Toba Batak', 'Urdu transliterate', 'Hindi transliterate', 'Urdu', 'Japanese', 'French', 'Marathi transliterate', 'Italian', 'Marathi', 'Bengali transliterate', 'Korean', 'Bengali', 'Spanish', 'Hindi', 'Indonesian', 'Simplified Chinese'}


In [11]:
language_codes = {
    "Toba Batak": "id",            # No dedicated ISO-639-1; Google uses Indonesian
    "Urdu": "ur",
    "Hindi": "hi",
    "Japanese": "ja",
    "French": "fr",
    "Marathi": "mr",
    "Italian": "it",
    "Bengali": "bn",
    "Korean": "ko",
    "Spanish": "es",
    "Indonesian": "id",
    "Simplified Chinese": "zh-cn"
}


In [7]:
from tqdm import tqdm
from googletrans import Translator
import time
import re
output_dir_translated = "language_texts_translated"
os.makedirs(output_dir_translated, exist_ok=True)

MAX_CHARS = 4500

def clean_text(text):
    """Remove common web noise from scraped text"""
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove common navigation/footer text patterns
    text = re.sub(r'(Home|Archive|Travel|Index|About Us|©.*)', '', text, flags=re.IGNORECASE)
    return text.strip()

lang_code = "en"
output_dir_lang = "language_texts"
output_dir_translated_lang = os.path.join(output_dir_translated, lang_code)   
for filename in tqdm(os.listdir(output_dir_lang)):
    file_path = os.path.join(output_dir_lang, filename)
    translated_file_path = os.path.join(output_dir_translated_lang, filename)
    
    if os.path.exists(translated_file_path):
        continue
        
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    # Clean the text first
    text = clean_text(text)
    os.makedirs(output_dir_translated_lang, exist_ok=True)
    with open(translated_file_path, "w", encoding="utf-8") as f:
        f.write(text)

100%|██████████| 2419/2419 [03:20<00:00, 12.08it/s] 


In [16]:
from tqdm import tqdm
from googletrans import Translator
import time
import re

translator = Translator()
output_dir_translated = "language_texts_translated"
os.makedirs(output_dir_translated, exist_ok=True)

MAX_CHARS = 4500

def clean_text(text):
    """Remove common web noise from scraped text"""
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove common navigation/footer text patterns
    text = re.sub(r'(Home|Archive|Travel|Index|About Us|©.*)', '', text, flags=re.IGNORECASE)
    return text.strip()

for key, lang_code in language_codes.items():
    print(f"Translating files to {key} ({lang_code})")
    output_dir_translated_lang = os.path.join(output_dir_translated, lang_code)
    os.makedirs(output_dir_translated_lang, exist_ok=True)
    
    for filename in tqdm(os.listdir(output_dir_lang)):
        file_path = os.path.join(output_dir_lang, filename)
        translated_file_path = os.path.join(output_dir_translated_lang, filename)
        
        if os.path.exists(translated_file_path):
            continue
            
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
        
        # Clean the text first
        text = clean_text(text)
        
        if not text or len(text) < 50:  # Skip very short texts
            continue
            
        try:
            if len(text) > MAX_CHARS:
                chunks = [text[i:i+MAX_CHARS] for i in range(0, len(text), MAX_CHARS)]
                translated_chunks = []
                for chunk in chunks:
                    result = translator.translate(chunk, dest=lang_code)
                    if result and result.text:
                        translated_chunks.append(result.text)
                    time.sleep(0.5)
                translated_text = " ".join(translated_chunks)
            else:
                result = translator.translate(text, dest=lang_code)
                translated_text = result.text if result else ""
            
            if translated_text:
                with open(translated_file_path, "w", encoding="utf-8") as f:
                    f.write(translated_text)
            
            time.sleep(0.3)
            
        except Exception as e:
            print(f"\nFailed to translate {filename} to {key}: {e}")
            time.sleep(2)

Translating files to Toba Batak (id)


  0%|          | 8/2419 [00:56<4:43:52,  7.06s/it]


KeyboardInterrupt: 

In [ ]:
# from tqdm import tqdm
# # Now we need to translate these files into diff langauges using googletrans
# from googletrans import Translator
# translator = Translator()
# output_dir_translated = "language_texts_translated"
# os.makedirs(output_dir_translated, exist_ok=True)

# for key, lang_code in language_codes.items():
#     print(f"Translating files to {key} ({lang_code})")
#     output_dir_translated_lang = os.path.join(output_dir_translated, lang_code)
#     os.makedirs(output_dir_translated_lang, exist_ok=True)
    
#     for filename in tqdm(os.listdir(output_dir_lang)):
#         file_path = os.path.join(output_dir_lang, filename)
#         with open(file_path, "r", encoding="utf-8") as f:
#             text = f.read()
        
#         try:
#             # Translate text to the correct language
#             translated = translator.translate(text, dest=lang_code)
#             translated_text = translated.text

#             # Save translated text to the correct language folder
#             translated_file_path = os.path.join(output_dir_translated_lang, filename)
#             with open(translated_file_path, "w", encoding="utf-8") as f:
#                 f.write(translated_text)
#         except Exception as e:
#             print(f"Failed to translate {filename} to {key}: {e}")

Translating files to Toba Batak (id)


  0%|          | 0/2419 [00:00<?, ?it/s]

Failed to translate 1053.txt to Toba Batak: the JSON object must be str, bytes or bytearray, not NoneType


  0%|          | 1/2419 [00:07<5:20:22,  7.95s/it]


KeyboardInterrupt: 